# Exercise — Configure Governed Access in Lake Formation

**Trailhead Provisions** needs four personas to have exactly the access below on the `customer`
table. Configure the Lake Formation grants with the provided helpers, then run the verification
cell to prove enforcement by querying as each persona in Athena. See `INSTRUCTIONS.md`.

| Persona | Intended access | LF mode |
|---|---|---|
| `co_marketing_analyst` | Customer **minus** `full_name`, `email`, `phone` | column-level |
| `co_cs_rep` | All columns, **rows where `region='EU'`** only | row-level |
| `co_data_steward` | Everything tagged `governed=true` | tag-based (LF-TBAC) |
| `co_auditor` | Schema + tags only; **no row data** | metadata-only |

> This exercise runs exclusively against **live AWS Glue, Lake Formation, and Amazon Athena** (no local fallback).

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog, S3 tables, and Lake Formation enforcement are seeded.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__}")
print(f"  Glue DB:          {gc.database}")
print(f"  Region:           {gc.region}")
print(f"  Athena workgroup: {gc.athena_workgroup}")
print(f"  Results bucket:   {gc.results_bucket}")

Backend: live -> AwsGovernedCatalog
  Glue DB:          trailhead
  Region:           us-east-1
  Athena workgroup: trailhead
  Results bucket:   s3://123456789012-trailhead-athena-results/


## 1. Reset & confirm the starting state (provided)
Clear any grants from a previous run, then confirm every persona starts with no access.

In [3]:
gc.reset()
gc.grants_summary()

,role,table,access
0,co_marketing_analyst,(none),no grants
1,co_cs_rep,(none),no grants
2,co_data_steward,(none),no grants
3,co_auditor,(none),no grants


## 2. Apply the four Lake Formation grants
Use the provided helpers:
- **Column-level (`co_marketing_analyst`)**: `gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])`
- **Row-level (`co_cs_rep`)**: `gc.create_row_filter(name, "customer", "region = 'EU'")` + `gc.grant_rows("co_cs_rep", "customer", name)`
- **Tag-based (`co_data_steward`)**: First tag `customer` as `governed=true` with `gc.assign_tag('customer', 'governed', 'true')` (the governed table in this exercise), then grant `co_data_steward` access by tag with `gc.grant_by_tag('co_data_steward', 'governed', 'true')`.
- **Metadata-only (`co_auditor`)**: `gc.grant_describe_only("co_auditor")`

In [4]:
# 1) column-level: analyst sees customer minus all three PII columns
gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])

# 2) row-level: cs_rep sees only EU rows
gc.create_row_filter("eu_only", "customer", "region = 'EU'")
gc.grant_rows("co_cs_rep", "customer", "eu_only")

# 3) tag-based: First tag `customer` as `governed=true` with `gc.assign_tag('customer', 'governed', 'true')`
#    (the governed table in this exercise), then grant `co_data_steward` access by tag with
#    `gc.grant_by_tag('co_data_steward', 'governed', 'true')`.
gc.assign_tag("customer", "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")

# 4) metadata-only: auditor gets DESCRIBE, no SELECT
gc.grant_describe_only("co_auditor")

gc.grants_summary()

,role,table,access
0,co_marketing_analyst,customer,cols=6 of 9; rows=all
1,co_cs_rep,customer,cols=all; rows=region = 'EU'
2,co_data_steward,customer,via LF-Tag (all cols/rows)
3,co_auditor,(all),DESCRIBE only (no rows)


## 3. Verify enforcement (provided)
Queries the governed table as each persona and asserts the intended outcome. Every check must read **PASS**.

In [5]:
results = verify_step7(gc)
print("All checks pass?", (results["result"] == "PASS").all())
results

All checks pass? True


,persona,check,result,backend,detail
0,co_marketing_analyst,"PII columns masked (full_name, email, phone)",PASS,live (AWS Lake Formation + Athena),"columns=['customer_id', 'region', 'country', '..."
1,co_cs_rep,"row filter region='EU' (all cols, 62 EU rows)",PASS,live (AWS Lake Formation + Athena),"cols=9/9, regions=['EU'], rows=62"
2,co_data_steward,tag grant (governed=true) = full access,PASS,live (AWS Lake Formation + Athena),"table_tagged=True, tag_grant=True, no_direct_g..."
3,co_auditor,"metadata-only (DESCRIBE ok, SELECT denied)",PASS,live (AWS Lake Formation + Athena),"describe_ok=True, select_denied=True"


## 4. Policy spec
Write a short policy spec documenting the intended access for each persona and the LF grant type used.

### Policy spec — Trailhead customer access

- **`co_marketing_analyst`** targets campaigns and must never see personal or contact PII. **Column-level**
  grant on `customer` excluding `full_name`, `email`, and `phone`; all 6 non-PII columns (`customer_id`,
  `loyalty_tier`, `region`, `country`, `marketing_consent`, `created_ts`) remain available.
- **`co_cs_rep`** handles EU support tickets only: full customer detail (all 9 columns) but **only
  `region = 'EU'` rows** (62 rows), via a **row-level** Lake Formation DataCellsFilter (`eu_only`).
  No access to NA or APAC customer rows.
- **`co_data_steward`** owns customer-domain quality and master-data resolution: full access via an
  **LF-Tag** grant on `governed=true` (`assign_tag('customer', 'governed', 'true')` +
  `grant_by_tag('co_data_steward', 'governed', 'true')`), so any future table tagged `governed=true`
  is covered automatically without per-table re-granting.
- **`co_auditor`** verifies governance posture and must **not** read customer row data: **metadata-only**
  (`DESCRIBE` on schema + tags via Glue), with `SELECT` in Athena denied. Least privilege — proves
  governance exists without touching a single personal record.

Each grant is the smallest privilege that satisfies the role's job; the auditor's metadata-only
access is the clearest example — visibility into structure with zero exposure of personal data.

---
## 5. Infrastructure as Code — Read and Explain

Open `terraform/governed-data-product/main.tf` (provided, read-only). This Terraform module
packages everything you just did by hand into a repeatable, version-controlled declaration.

**Your task**: Read the module and answer the questions below. You do NOT need to run
`terraform apply` — this is a reading-comprehension exercise.

### Questions

1. **What is the governance chain?** List the 4 AWS resources the module creates, in order,
   and explain how each one connects to the next to enforce governed access.

2. **Why is there NO `s3:GetObject` on the data bucket?** What does the consumer role use
   instead to read data, and what enforces the column/row restrictions?

3. **How does `excluded_column_names` in the `aws_lakeformation_permissions` resource relate
   to what you did in Section 2?** Which of your four grants does this Terraform resource
   implement?

4. **If a new domain team wants to add a `product` data product with PII column
   `supplier_contact_email`, what values would they pass to this module?** (List the
   variable values.)

### Answers

**1. The governance chain (S3 → Glue → Lake Formation → IAM):**

| # | Resource | Purpose | Connection to next |
|---|----------|---------|--------------------|
| 1 | `aws_s3_bucket.data` | Physical storage for the data product (Parquet/CSV files) | Registered with Lake Formation via `aws_lakeformation_resource.data` so LF mediates all data access |
| 2 | `aws_glue_catalog_table.table` | Logical catalog entry — schema, `owner`, `classification`, `retention` metadata | Lake Formation attaches the `governed=true` LF-Tag (`aws_lakeformation_resource_lf_tag.tag_table`) and reads table/column metadata |
| 3 | `aws_lakeformation_permissions.consumer_columns` | Fine-grained grant — column-level `SELECT` with `excluded_column_names = var.pii_columns` | Applied to the IAM consumer role (`aws_iam_role.consumer.arn`), enforced at query time |
| 4 | `aws_iam_role.consumer` | The identity that queries the data product | Has Athena + Glue read + `lakeformation:GetDataAccess` (`aws_iam_role_policy.consumer`) but NO direct S3 data bucket access |

The chain enforces governance because the consumer can only reach data through Athena, Athena
consults Lake Formation for permissions, and LF checks the grant that excludes PII columns.
No bypass path exists because the IAM policy has no `s3:GetObject` on the data bucket.

**2. No `s3:GetObject` on data:**

The consumer role gets `s3:GetObject` and `s3:PutObject` ONLY on the Athena results bucket
(`arn:aws:s3:::trailhead-athena-results/*`), not on `aws_s3_bucket.data`. To read data, the
consumer calls `athena:StartQueryExecution` + `lakeformation:GetDataAccess`. Lake Formation
evaluates the grant (column exclusions, row filters, tag membership) and vends temporary, scoped
S3 credentials to Athena — the consumer never reads raw S3 files directly.

**3. Relationship to Section 2:**

`excluded_column_names = var.pii_columns` in `aws_lakeformation_permissions.consumer_columns` is the
Terraform equivalent of our
`gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])` — it
implements the **column-level grant** (grant type 1 of our four). The module declares this default
consumer column mask plus the `governed=true` LF-Tag; row-level DataCellsFilters and metadata-only
auditor grants would be added as additional `aws_lakeformation_data_cells_filter` and
`aws_lakeformation_permissions` resources.

**4. Module call for `product` with PII:**

```hcl
module "product" {
  source              = "./governed-data-product"
  product_name        = "product"
  domain              = "Inventory & Fulfillment"
  pii_columns         = ["supplier_contact_email"]
  data_lake_admin_arn = "arn:aws:iam::123456789012:role/LakeFormationAdmin"
}
```

---
# Solution — Configure Governed Access in Lake Formation

## Summary

This exercise practices **fine-grained access as a governance control**: implementing column-,
row-, tag-, and metadata-level access in **live AWS Lake Formation** and **verifying** it by
assuming each persona role via STS and querying in **Amazon Athena** and **AWS Glue** (no local
fallback). You also analyze how the `S3 -> Glue -> Lake Formation -> IAM` governance chain is
declared in Terraform.

## Key steps

```python
gc.reset()
gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])  # column-level
gc.create_row_filter("eu_only", "customer", "region = 'EU'")                                   # row-level
gc.grant_rows("co_cs_rep", "customer", "eu_only")
gc.assign_tag("customer", "governed", "true")                                                  # tag-based (LF-TBAC)
gc.grant_by_tag("co_data_steward", "governed", "true")
gc.grant_describe_only("co_auditor")                                                           # metadata-only
results = verify_step7(gc)        # every row PASS on live (AWS Lake Formation + Athena)
```

Expected verification:

| Persona | Check | Result | Backend |
|---|---|---|---|
| `co_marketing_analyst` | `PII columns masked (full_name, email, phone)` | PASS | `live (AWS Lake Formation + Athena)` |
| `co_cs_rep` | `row filter region='EU' (all cols, 62 EU rows)` | PASS | `live (AWS Lake Formation + Athena)` |
| `co_data_steward` | `tag grant (governed=true) = full access` | PASS | `live (AWS Lake Formation + Athena)` |
| `co_auditor` | `metadata-only (DESCRIBE ok, SELECT denied)` | PASS | `live (AWS Lake Formation + Athena)` |

## Common Mistakes to Avoid

- **Excluding only `email` and `phone` for `co_marketing_analyst`** — `full_name` is also a PII
  column on `customer`; all three (`["full_name", "email", "phone"]`) must be excluded.
- **Granting the steward per-table instead of by tag** — first tag `customer` with
  `gc.assign_tag("customer", "governed", "true")`, then grant by tag with
  `gc.grant_by_tag("co_data_steward", "governed", "true")` so new governed tables inherit access
  automatically.
- **Giving the auditor SELECT "just to be safe"** — the point is metadata-only (`DESCRIBE` via Glue);
  `SELECT` in Athena must be denied.
- **Forgetting the row filter is row-level, not a column grant** — `co_cs_rep` keeps all 9 columns
  on `customer`, filtered to the 62 rows where `region = 'EU'`.
- **Declaring access without verifying** — the graded skill is the passing `verify_step7(gc)` output,
  which proves live Lake Formation and Athena enforce the policy.